In [1]:
CARDS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]  # 11 represents an Ace
cards_length = len(CARDS)

import random
def draw_card():
    return random.choice(CARDS)

In [ ]:
from collections import defaultdict
import random

ACTIONS = ("H", "S")
DEALER_UPCARDS = (2, 3, 4, 5, 6, 7, 8, 9, 10, 11)


def hand_value(cards):
    """Return the best blackjack total and whether the hand has a usable ace."""
    total = sum(cards)
    aces = cards.count(11)
    while total > 21 and aces:
        total -= 10
        aces -= 1
    return total, aces > 0


def dealer_total(dealer_upcard):
    """Deal a dealer hand and return its final total, or 22 for a bust."""
    cards = [dealer_upcard, draw_card()]
    total, usable_ace = hand_value(cards)
    while total < 17:
        cards.append(draw_card())
        total, usable_ace = hand_value(cards)
    return 22 if total > 21 else total


def settle(player_total, dealer_final_total):
    if player_total > 21:
        return -1
    if dealer_final_total > 21 or player_total > dealer_final_total:
        return 1
    if player_total < dealer_final_total:
        return -1
    return 0


def add_card(total, usable_ace, card):
    total += card
    if card == 11:
        if total <= 21:
            usable_ace = True
        else:
            total -= 10
    if total > 21 and usable_ace:
        total -= 10
        usable_ace = False
    return total, usable_ace


def rollout(state, first_action, policy):
    """Simulate one game from a state after forcing its first action."""
    total, usable_ace, dealer_upcard = state
    action = first_action

    while True:
        if action == "S":
            return settle(total, dealer_total(dealer_upcard))

        total, usable_ace = add_card(total, usable_ace, draw_card())
        if total > 21:
            return -1

        next_state = (total, usable_ace, dealer_upcard)
        action = policy.get(next_state, "H")


def monte_carlo_policy_iteration():
    """Estimate action values by simulation and improve the hit/stand policy."""
    random.seed(42)
    states = [
        (total, usable_ace, dealer_upcard)
        for usable_ace, totals in ((False, range(4, 22)), (True, range(13, 22)))
        for total in totals
        for dealer_upcard in DEALER_UPCARDS
    ]
    policy = {state: "H" for state in states}
    q_values = defaultdict(dict)

    for _ in range(10):
        for state in states:
            for action in ACTIONS:
                SIMULATIONS_PER_ACTION = 5000
                rewards = [rollout(state, action, policy) for _ in range(SIMULATIONS_PER_ACTION)]
                q_values[state][action] = sum(rewards) / SIMULATIONS_PER_ACTION

        updated_policy = {
            state: max(ACTIONS, key=q_values[state].get) for state in states
        }
        if updated_policy == policy:
            break
        policy = updated_policy

    return policy, q_values


def print_strategy_table(policy, usable_ace):
    label = "SOFT" if usable_ace else "HARD"
    totals = range(13, 22) if usable_ace else range(4, 22)
    print(f"\\n{label} HAND STRATEGY (H = hit, S = stand)")
    print("Total   " + " ".join(f"{up:>2}" for up in DEALER_UPCARDS))
    for total in totals:
        actions = [policy[(total, usable_ace, upcard)] for upcard in DEALER_UPCARDS]
        print(f"{total:>5}   " + " ".join(f"{action:>2}" for action in actions))


policy, q_values = monte_carlo_policy_iteration()
print_strategy_table(policy, usable_ace=False)
print_strategy_table(policy, usable_ace=True)

example_state = (16, False, 10)
print("\\nHard 16 vs dealer 10:", q_values[example_state])


[(4, False, 2), (4, False, 3), (4, False, 4), (4, False, 5), (4, False, 6), (4, False, 7), (4, False, 8), (4, False, 9), (4, False, 10), (4, False, 11), (5, False, 2), (5, False, 3), (5, False, 4), (5, False, 5), (5, False, 6), (5, False, 7), (5, False, 8), (5, False, 9), (5, False, 10), (5, False, 11), (6, False, 2), (6, False, 3), (6, False, 4), (6, False, 5), (6, False, 6), (6, False, 7), (6, False, 8), (6, False, 9), (6, False, 10), (6, False, 11), (7, False, 2), (7, False, 3), (7, False, 4), (7, False, 5), (7, False, 6), (7, False, 7), (7, False, 8), (7, False, 9), (7, False, 10), (7, False, 11), (8, False, 2), (8, False, 3), (8, False, 4), (8, False, 5), (8, False, 6), (8, False, 7), (8, False, 8), (8, False, 9), (8, False, 10), (8, False, 11), (9, False, 2), (9, False, 3), (9, False, 4), (9, False, 5), (9, False, 6), (9, False, 7), (9, False, 8), (9, False, 9), (9, False, 10), (9, False, 11), (10, False, 2), (10, False, 3), (10, False, 4), (10, False, 5), (10, False, 6), (10, Fa